<img src="https://raw.githubusercontent.com/pasdptt/public_teaching_tools4DS/main/assets/brand/logo-160.png" width="80" align="right" alt="Course logo">

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pasdptt/public_teaching_tools4DS/blob/main/modules/m08-smoothing-temporal/lab/lab8_smoothing_temporal.ipynb)

# Lab 8 — Smoothing, aggregation and temporal evaluation

In [ ]:
# Setup: run this cell first (Google Colab or a local Jupyter kernel).
COURSE_REF = "main"  # version of the course repository to install the helper package from
try:
    import tsds
except ImportError:  # first run in a fresh Colab runtime: install the course helper package
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    f"tsds @ git+https://github.com/pasdptt/public_teaching_tools4DS@{COURSE_REF}#subdirectory=src"], check=True)
    import tsds

tsds.env.check()          # prints versions; warns if a package is outside the tested range
SEED = tsds.seed_everything()
rng = tsds.rng()

import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.holtwinters import SimpleExpSmoothing, ExponentialSmoothing
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, cross_val_predict
warnings.filterwarnings("ignore", module="statsmodels")

**Course:** 040563107 Tools for Statistical Data Science · **Module m08** · **Time:** 3 hours · **Graded lab G5** (3⅓% of the course) · includes an **AI-audit task**

**After this lab you can:**
* recode a sentinel value and aggregate irregular hourly records to a regular daily series with a stated coverage rule (objective m08-O4);
* distinguish trailing from centred smoothers (O1);
* implement a kernel smoother and choose its bandwidth by a stated criterion, and see why leave-one-out fails on autocorrelated data (O3);
* evaluate one-step forecasts by rolling-origin validation against naive baselines (O2, O5);
* audit an AI-generated forecast evaluation that leaks the future (O5, AI audit).

**Two steps:** in the lab session we **run through** the ideas together. Then, as homework, you **do this lab on your own**
from a fresh copy and upload the `.ipynb` to the G5 assignment in Microsoft Teams. **That homework is what is marked.**

**How it is marked** ([lab rubric](https://github.com/pasdptt/public_teaching_tools4DS/blob/main/course/rubrics.md#lab-rubric-graded-labs)):
your notebook is re-run from top to bottom in a fresh session and checked with hidden tests (correctness 3 points,
reproducibility 1 point); staff mark your written answers (failure analysis 2, interpretation 2).

**How to work:**
1. Run the setup cell first.
2. Complete every cell marked **Task**. Replace `raise NotImplementedError()` with your code, then run the **Self-check** cell below it; it prints `OK` when your answer passes. Do not edit given cells or self-check cells.
3. Write the answers marked **Q** after *Your answer:* (at most 100 words each).
4. Before submitting, use *Runtime → Restart session and run all*. The notebook must run top to bottom without errors.
5. Fill in the **AI-use log** at the end.

## Part 1 — Sentinels and aggregation (about 35 minutes)

The Air Quality data are hourly readings from a multi-sensor device in an Italian city, March 2004 to April 2005. **Missing
values are coded as −200** (check the dataset's documentation: `tsds.data.info("air_quality")`). The file also ends with empty
rows. Date and time are separate text columns, e.g. `10/03/2004` (day/month/year) and `18.00.00`.

First, what happens if the sentinel is treated as a measurement? The course's old notebook smoothed temperature this way
(audit issue M15).

In [ ]:
raw = tsds.data.load("air_quality")
print(raw.shape)
T_old = pd.to_numeric(raw["T"]).dropna()   # temperature, sentinels left in
ema_old = T_old.ewm(alpha=0.1).mean()
print(f"sentinel rows in T: {(T_old == -200).sum()} | EWMA of T with sentinels: min {ema_old.min():.1f} °C")

An exponentially weighted moving average of temperature that falls below −190 °C is not a smoother of anything real. Every
smoother, aggregate and forecast must start from **recoded** data.

**Aggregation.** Hourly readings with gaps are not a regular series. We aggregate to a **daily mean**, but only for days with
enough valid hours: a day with 3 valid hours does not represent the day. Isolated missing days inside the series are then filled
by time interpolation (and counted); incomplete days at the two ends are dropped.

In [ ]:
# Task T1
# daily_series(raw, col, min_hours=18): from the raw table,
#   (a) drop rows whose Date is missing; build a DatetimeIndex from Date + " " + Time with format "%d/%m/%Y %H.%M.%S"
#   (b) take column col as numbers and recode -200 to NaN
#   (c) daily mean (resample("D")) of the valid hours; set a day to NaN if it has fewer than min_hours valid hours
#   (d) fill NaN days inside the series with .interpolate("time", limit_area="inside"), then drop remaining NaN (the ends)
#   Return a pandas Series with a daily DatetimeIndex.
# Then: y = daily_series(raw, "C6H6(GT)")   (benzene, micrograms per cubic metre)
def daily_series(raw, col, min_hours=18):
    # YOUR CODE HERE
    raise NotImplementedError()


# YOUR CODE HERE
raise NotImplementedError()
print(f"{len(y)} days, {y.index[0].date()} to {y.index[-1].date()}; mean {y.mean():.2f}, min {y.min():.2f}, max {y.max():.2f}")
print("mean by weekday (0 = Monday):", y.groupby(y.index.day_of_week).mean().round(1).to_dict())

In [ ]:
# Self-check for task T1: run it; it should print OK.
assert isinstance(y.index, pd.DatetimeIndex) and set(np.diff(y.index.to_numpy())) == {np.timedelta64(1, "D")}, "a regular daily index"
assert len(y) == 389 and y.isna().sum() == 0
assert (y > 0).all(), "sentinels must be gone"
assert str(y.index[0].date()) == "2004-03-11" and str(y.index[-1].date()) == "2005-04-03"
print("OK")

## Part 2 — Smoothers and the bandwidth (about 40 minutes)

**Trailing vs centred.** A trailing 7-day mean uses days $t-6, \dots, t$: it is available at the end of day $t$, so it can be
used for forecasting, but it **lags** a changing level. A centred 7-day mean uses $t-3, \dots, t+3$: better aligned, but it uses
**the future**, so it can describe the past but never forecast.

**Kernel (Nadaraya–Watson) smoother.** $\hat m(t) = \sum_j K\big(\tfrac{t - t_j}{h}\big) y_j \big/ \sum_j K\big(\tfrac{t-t_j}{h}\big)$
with a Gaussian kernel $K(u) = e^{-u^2/2}$ and **bandwidth** $h$ (days). Small $h$ follows the noise (low bias, high variance);
large $h$ flattens real changes (high bias, low variance).

**Choosing $h$ by a stated criterion.** Predict each day from the *other* days and pick the $h$ with the smallest mean squared
error. Leave-one-out leaves out only day $t$ itself. But neighbouring days are strongly correlated, so they "know" day $t$, and
leave-one-out rewards following the noise. Leaving out a **block** (all days within $r$ of $t$) is a more honest criterion.

In [ ]:
H_GRID = [1, 2, 3, 5, 7, 10, 14, 21, 30]   # bandwidths to compare, in days

In [ ]:
# Task T2
# (a) trailing7 = 7-day trailing mean of y; centred7 = 7-day centred mean of y (pandas rolling; keep the NaN at the edges)
# (b) nw_smooth(values, h, radius=None): Gaussian-kernel smoother at every position i = 0..n-1 using positions 0..n-1 as time.
#     radius=None: use all points; radius=r (an integer >= 0): give weight 0 to every point j with |i - j| <= r
#     (r = 0 is leave-one-out). Return a numpy array of length n.
# (c) cv_loo:   dict h -> mean squared error of v against nw_smooth(v, h, radius=0), for h in H_GRID
#     cv_block: the same with radius=3 (leave out a week centred on each day)
#     h_loo, h_block: the h with the smallest error in each
v = y.to_numpy()
# YOUR CODE HERE
raise NotImplementedError()


def nw_smooth(values, h, radius=None):
    # YOUR CODE HERE
    raise NotImplementedError()


# YOUR CODE HERE
raise NotImplementedError()
print("leave-one-out:", {h: round(e, 2) for h, e in cv_loo.items()}, "-> h =", h_loo)
print("leave-a-week-out:", {h: round(e, 2) for h, e in cv_block.items()}, "-> h =", h_block)
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(y.index, v, lw=0.8, color="0.6", label="daily mean")
ax.plot(y.index, trailing7, label="trailing 7-day mean")
ax.plot(y.index, centred7, label="centred 7-day mean")
ax.plot(y.index, nw_smooth(v, h_block), lw=2.5, label=f"kernel smoother, h = {h_block}")
ax.set_ylabel("benzene"); ax.legend(); plt.show()

In [ ]:
# Self-check for task T2: run it; it should print OK.
assert trailing7.isna().sum() == 6 and centred7.isna().sum() == 6
assert abs(trailing7.iloc[6] - y.iloc[:7].mean()) < 1e-9 and abs(centred7.iloc[3] - y.iloc[:7].mean()) < 1e-9
s = nw_smooth(v, 5); assert s.shape == v.shape and abs(s.mean() - v.mean()) < 0.5
assert np.allclose(nw_smooth(np.full(20, 4.0), 3, radius=2), 4.0), "a constant series must smooth to itself"
assert h_loo == 1 and h_block == 21, (h_loo, h_block)
print("OK")

**Q1.** **Interpretation.** Compare the trailing and the centred 7-day means in your plot: which lags, and which could be used to
forecast tomorrow? Then explain why leave-one-out and leave-a-week-out choose such different bandwidths, and which you would
report as "the smoothing parameter, chosen by …".

*Your answer:* 

## Part 3 — Forecasts evaluated by rolling origin (about 45 minutes)

To evaluate a **one-day-ahead forecast**, imitate how it would be used: at each origin $o$, fit using days $0, \dots, o-1$ only,
forecast day $o$, record the error, move the origin forward one day. The first 75% of the series is the initial training window.

Methods (each a function from the history to one forecast):
* **naive:** tomorrow = today; **seasonal naive:** tomorrow = the same weekday last week;
* **SES:** simple exponential smoothing with $\alpha$ **estimated** from the history (not fixed at 0.6, as in the old notebook, audit M18);
* **Holt–Winters (weekly):** exponential smoothing with an additive weekly season.

In [ ]:
forecasters = {
    "naive": lambda h: h[-1],
    "seasonal_naive": lambda h: h[-7],
    "ses": lambda h: SimpleExpSmoothing(h, initialization_method="estimated").fit().forecast(1)[0],
    "hw_weekly": lambda h: ExponentialSmoothing(h, seasonal="add", seasonal_periods=7,
                                                initialization_method="estimated").fit().forecast(1)[0],
}

In [ ]:
# Task T3
# (a) rolling_origin(values, forecaster, first): for o = first, first+1, ..., n-1, call forecaster(values[:o]) and record
#     values[o] - forecast. Return a numpy array of the n - first errors.
# (b) mae: dict name -> mean absolute error of rolling_origin(v, f, FIRST), for every name, f in forecasters
# (c) ses_alpha: the estimated alpha of SimpleExpSmoothing fitted to v[:FIRST] (fit().params["smoothing_level"])
FIRST = int(0.75 * len(v))
print("initial training window:", FIRST, "days; test origins:", len(v) - FIRST)


def rolling_origin(values, forecaster, first):
    # YOUR CODE HERE
    raise NotImplementedError()


# YOUR CODE HERE
raise NotImplementedError()
print("MAE:", {k: round(e, 3) for k, e in mae.items()}, "| SES alpha:", round(ses_alpha, 3))

In [ ]:
# Self-check for task T3: run it; it should print OK.
e = rolling_origin(np.arange(10.0), lambda h: h[-1], 6)
assert np.allclose(e, 1.0) and len(e) == 4, "naive on 0, 1, ..., 9 misses by exactly 1 each time"
assert set(mae) == set(forecasters) and abs(mae["naive"] - 2.617) < 0.01
assert mae["hw_weekly"] < mae["naive"] < mae["seasonal_naive"]
assert 0.8 < ses_alpha <= 1.0
print("OK")

**Q2.** **Interpretation.** Rank the four methods by MAE. What does the estimated $\alpha$ say about SES on this series, and why does
the weekly Holt–Winters model beat both naive baselines? Use the weekday means from Part 1.

*Your answer:* 

## Part 4 — AI audit: a forecast that is too good (about 30 minutes)

A student asked an AI assistant for *"a regression model to forecast tomorrow's daily benzene, with cross-validation"*. The code below is
representative of what such tools produce; the task cell below runs it on your series.

In [ ]:
# ---- AI-generated code (do not edit) ----
def ai_forecast_evaluation(raw, series):
    data = raw.dropna(subset=["Date"]).copy()
    data.index = pd.to_datetime(data["Date"] + " " + data["Time"], format="%d/%m/%Y %H.%M.%S")
    data = data.replace(-200, np.nan)
    sensor = data["PT08.S2(NMHC)"].resample("D").mean().reindex(series.index).interpolate()
    df = pd.DataFrame({"benzene": series.to_numpy(), "sensor_today": sensor.to_numpy()}, index=series.index)
    df["rolling_mean_3"] = df["benzene"].rolling(3, center=True, min_periods=1).mean()
    X_ai, y_ai = df[["sensor_today", "rolling_mean_3"]], df["benzene"]
    pred_ai = cross_val_predict(LinearRegression(), X_ai, y_ai, cv=KFold(n_splits=5, shuffle=True, random_state=42))
    mae = float(np.mean(np.abs(y_ai - pred_ai)))
    print(f"Forecast MAE: {mae:.3f}  -> beats every baseline!")
    return mae
# ---- end of AI-generated code ----

In [ ]:
# Task T4
mae_ai = ai_forecast_evaluation(raw, y)   # run the AI-generated evaluation on your daily series

# An honest version that could really be used each evening to forecast tomorrow:
# (a) feats: a DataFrame with the same index as y and columns
#       lag1  = benzene one day earlier          (y.shift(1))
#       lag7  = benzene seven days earlier        (y.shift(7))
#       mean7 = mean of the 7 days BEFORE the day (y.shift(1).rolling(7).mean())
# (b) mae_honest: rolling-origin MAE with the same origins o = FIRST, ..., n-1: at each origin fit LinearRegression on the rows
#     7, ..., o-1 of feats (the first 7 rows have missing features) with targets v[7:o], and predict row o.
# YOUR CODE HERE
raise NotImplementedError()
print(f"AI-generated evaluation: {mae_ai:.3f} | honest rolling-origin evaluation: {mae_honest:.3f} | naive: {mae['naive']:.3f}")

In [ ]:
# Self-check for task T4: run it; it should print OK.
assert list(feats.columns) == ["lag1", "lag7", "mean7"] and feats.index.equals(y.index)
assert feats["mean7"].isna().sum() == 7 and abs(feats["mean7"].iloc[7] - y.iloc[:7].mean()) < 1e-9
assert mae_ai < 1.0 and 2.0 < mae_honest, "the honest error should be far worse than the AI's"
assert abs(mae_honest - 2.552) < 0.02, mae_honest
print("OK")

**Q3.** **AI audit.** List the problems in the AI-generated evaluation that make its MAE too good, using your numbers. For each, say
which line causes it and how your honest version avoids it.

*Your answer:* 

**Q4.** **Failure analysis.** Use Part 1 to explain why the old notebook's exponential smoothing of temperature fell below −190 °C, and
why exponential smoothing should not be fitted to the raw hourly rows either (the old notebook also fitted SES to transaction rows
with several records per day, audit M18). State the two checks you now run before smoothing any series.

*Your answer:* 

## AI-use log (required; see the course AI policy)

| Tool | What you asked or used it for | What you accepted or changed | How you verified it |
|---|---|---|---|
| *e.g. Colab AI* | *how to resample hourly data to daily means* | *added the 18-hour coverage rule myself* | *T1 self-check: 389 days, no sentinels* |
| | | | |

If you used no AI tool, write "None".

**Cite the data** when you use them: Vito, S. (2008). Air Quality [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C59K5F (CC BY 4.0).